# Worksheet 4, Part 3 — Abundance diagnostics (Fe I trends and pruning)
### HE 1002-0755

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
os.makedirs("results", exist_ok=True); os.makedirs("plots", exist_ok=True)

# >>> point this at your SMHR export from the coadd session <<<
EXPORT = "test_input/TEST_0543_spectral_model_measurements.txt"
IS_TEST = "TEST_" in os.path.basename(EXPORT)

t = Table.read(EXPORT, format="ascii.fixed_width_two_line")
t["REW"] = np.log10(t["EW"] * 1e-3 / t["wavelength"])          # SMHR exports EW in mA
print(f"{len(t)} lines from {EXPORT}" + ("   [TEST INPUT - not for the paper]" if IS_TEST else ""))
for sp in np.unique(t["species"]):
    print(f"  species {sp:5.1f}: {np.sum(t['species'] == sp)} lines")

plt.rcParams.update({"font.family": "serif", "font.size": 11, "xtick.direction": "in",
    "ytick.direction": "in", "xtick.top": True, "ytick.right": True, "legend.frameon": False})

## 1. Line regression with a slope uncertainty

In [ ]:
def fit_line(x, y):
    (b, a), cov = np.polyfit(x, y, 1, cov=True)
    return a, b, float(np.sqrt(cov[0, 0]))

def clip_iter(y, nsig=2.5):
    keep = np.isfinite(y)
    while True:
        m, s = y[keep].mean(), y[keep].std(ddof=1)
        new = keep & (np.abs(y - m) <= nsig*s)
        if new.sum() == keep.sum():
            return keep, m, s
        keep = new

## 2. Fe I: abundance vs excitation potential and vs REW

In [ ]:
fe1 = t[t["species"] == 26.0]
y = np.array(fe1["logeps"], float)
keep, m, s = clip_iter(y)
print(f"Fe I: N = {len(y)}, mean {np.nanmean(y):.2f}, sigma {np.nanstd(y, ddof=1):.2f}")
print(f"after iterative 2.5 sigma clip: N = {keep.sum()}, mean {m:.2f}, sigma {s:.2f}  -> [Fe/H] = {m - 7.50:+.2f}")

fits = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, col, lab in ((axes[0], "expot", r"excitation potential $\chi$ (eV)"), (axes[1], "REW", r"REW $=\log(\mathrm{EW}/\lambda)$")):
    x = np.array(fe1[col], float)
    ax.plot(x[~keep], y[~keep], "x", color="0.6", label=r"outside 2.5$\sigma$")
    ax.plot(x[keep], y[keep], "o", ms=4, color="C0", label="kept")
    for sel, ls, name in ((np.isfinite(y), ":", "all"), (keep, "-", "kept")):
        a, b, eb = fit_line(x[sel], y[sel])
        xx = np.linspace(x[sel].min(), x[sel].max(), 2)
        ax.plot(xx, a + b*xx, ls, color="C3", label=f"{name}: slope {b:+.3f} ± {eb:.3f}")
        fits[f"{col}_{name}"] = dict(slope=b, slope_err=eb, intercept=a, N=int(sel.sum()), significance=abs(b)/eb)
    ax.set_xlabel(lab); ax.legend(fontsize=8)
axes[0].set_ylabel(r"$\log\epsilon$(Fe I)")
if IS_TEST: fig.suptitle("TEST INPUT - single exposure, lines not yet checked", color="C3", fontsize=10)
fig.tight_layout(); fig.savefig("plots/feI_trends_HE1002-0755.pdf"); fig.savefig("plots/feI_trends_HE1002-0755.png", dpi=150)

for k, v in fits.items():
    print(f"{k:12s} slope {v['slope']:+.4f} ± {v['slope_err']:.4f}  ({v['significance']:.1f} sigma, N = {v['N']})")

## 3. Suggested clips for every species

In [ ]:
rows = []
for sp in np.unique(t["species"]):
    tt = t[t["species"] == sp]; yy = np.array(tt["logeps"], float)
    if len(tt) >= 4:
        k, m, s = clip_iter(yy)
        for r in tt[~k]:
            rows.append(dict(species=float(sp), wavelength=float(r["wavelength"]), logeps=float(r["logeps"]),
                             offset=float(r["logeps"] - m), EW=float(r["EW"]), REW=float(r["REW"]), reason="> 2.5 sigma"))
        print(f"{sp:5.1f}: N = {len(tt):3d}  mean {m:5.2f}  sigma {s:4.2f}  suggest unchecking {np.sum(~k)}")
    else:
        strong = np.sum(tt["REW"] > -4.5)
        print(f"{sp:5.1f}: N = {len(tt):3d}  too few lines for sigma clipping; {strong} of them with REW > -4.5 - check each by eye")
clip = Table(rows) if rows else Table(names=("species", "wavelength", "logeps", "offset", "EW", "REW", "reason"))
clip.write("results/suggested_clips.txt", format="ascii.fixed_width_two_line", overwrite=True)
json.dump(dict(export=EXPORT, test_input=IS_TEST, feI=fits), open("results/feI_trend_fits.json", "w"), indent=1)
print(f"\n{len(clip)} lines listed in results/suggested_clips.txt")
clip[:15] if len(clip) else None